# Phase 7 - Selective LoD2 feasibility

This notebook assesses all 61 Phase 6 building records, attempts robust roof-plane reconstruction only for supported non-boundary candidates, and creates a mixed LoD1/LoD2 model without inventing unsupported roof geometry.

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "pyproject.toml").exists():
    project_root = project_root.parent
if not (project_root / "pyproject.toml").exists():
    raise RuntimeError("Could not locate the project root containing pyproject.toml")

src_path = str(project_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import geopandas as gpd
import pandas as pd
import pyvista as pv
from IPython.display import JSON, Image, display

from uqgems.phase6 import _pyvista_mesh
from uqgems.phase7 import STATUS_HEX, _load_scene_meshes, run_phase7

project_root

## Run and validate

The initial run scans the analysis LAZ. A later run reuses the core model only when source, configuration and output hashes match; review tables and figures are refreshed.

In [ ]:
summary = run_phase7(project_root)
print(f"Phase 7 status: {summary['status']}")
print(f"Cached core products reused: {summary['cache_reused']}")
display(JSON(summary["checks"], expanded=True))

## Pilot result and decision gate

An LoD2 label means the roof passed point-coverage, residual, plane-partition and closed-solid gates. Internal residuals are not independent survey accuracy.

In [ ]:
statistics = summary["current_statistics"]
overview = pd.Series(
    {
        "buildings assessed": statistics["buildings_assessed"],
        "source-supported candidates": statistics["candidate_buildings"],
        "accepted LoD2 buildings": statistics["accepted_lod2_buildings"],
        "accepted roof surfaces": statistics["accepted_roof_surfaces"],
        "detected ridge segments": statistics["detected_ridge_segments"],
        "median accepted surface coverage": statistics["accepted_surface_coverage"]["median"],
        "median accepted surface RMSE (m)": statistics["accepted_surface_rmse_m"]["median"],
        "watertight mixed-model meshes": summary["presentation_mesh"]["watertight_geometry_count"],
    },
    name="value",
)
display(overview.to_frame())
display(pd.Series(statistics["status_counts"], name="buildings").to_frame())
print(summary["decision_gate"])

## Building-level evidence

The `building_status` layer covers every footprint. The 3D `roof_surfaces` and `ridge_lines` layers contain accepted LoD2 geometry only.

In [ ]:
model_path = project_root / summary["outputs"]["gpkg"]
buildings = gpd.read_file(model_path, layer="building_status")
roof_surfaces = gpd.read_file(model_path, layer="roof_surfaces")
ridge_lines = gpd.read_file(model_path, layer="ridge_lines")
columns = [
    "building_id",
    "building_name",
    "lod2_candidate",
    "lod2_status",
    "roof_plane_count",
    "surface_covered_fraction",
    "surface_rmse_m",
    "partition_consistency",
    "ridge_count",
    "lod2_reason",
]
display(buildings[columns].sort_values(["lod2_status", "building_id"]))
print(f"{len(roof_surfaces)} accepted roof surfaces; {len(ridge_lines)} ridge segments")

In [ ]:
review_path = project_root / summary["outputs"]["tables"]["manual_review"]
display(pd.read_csv(review_path))

## Fixed visual checks

The oblique figures use three-times vertical exaggeration for inspection only. Stored GeoPackage and GLB elevations remain at true scale.

In [ ]:
figure_order = [
    "feasibility",
    "roof_planes",
    "fit_examples",
    "residuals",
    "northeast",
    "southwest",
]
for name in figure_order:
    print(name.replace("_", " ").title())
    display(Image(filename=str(project_root / summary["outputs"]["figures"][name])))

## Interactive mixed-LoD scene

Rotate and zoom this local-coordinate building model directly in VS Code. Colours match the feasibility status map.

In [ ]:
meshes = _load_scene_meshes(project_root / summary["outputs"]["glb"])
indexed = buildings.set_index("building_id")
plotter = pv.Plotter(off_screen=True, notebook=True, window_size=(1100, 750))
plotter.set_background("#edf2f4")
for building_id, mesh in meshes.items():
    if building_id not in indexed.index:
        continue
    status = str(indexed.loc[building_id, "lod2_status"])
    plotter.add_mesh(_pyvista_mesh(mesh), color=STATUS_HEX[status], smooth_shading=False)
plotter.view_isometric()
plotter.show_axes()
interactive_viewer = plotter.show(jupyter_backend="html", return_viewer=True)
interactive_viewer

## Phase 8 handoff

Use the mixed GLB for presentation and the GeoPackage for GIS analysis. Do not expand automated LoD2 without retaining these gates. Phase 8 can standardise authorised utility data against EPSG:7856/AHD while preserving accuracy, sensitivity and source metadata. Read `PHASE7_LOD2_FEASIBILITY.md` before presenting the model as current or authoritative.